In [5]:
from keras.models import Model 
from keras.layers import Input, Convolution2D, MaxPooling2D, Dense, Dropout, Flatten, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
# import np_utils
import numpy as np
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import os
import datetime
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

In [11]:
poor = pd.read_csv("C:/Users/met48/Desktop/TS-Clustering/SimData/bank_reserves_outputs_poor.csv", header=None)
middle = pd.read_csv("C:/Users/met48/Desktop/TS-Clustering/SimData/bank_reserves_outputs_middle.csv", header=None)
rich = pd.read_csv("C:/Users/met48/Desktop/TS-Clustering/SimData/bank_reserves_outputs_rich.csv", header=None)
sc = StandardScaler()
bank_reserves = []
for i in np.arange(0, poor.shape[0]):
    sample = pd.concat([poor.iloc[i], middle.iloc[i]], axis=0).T
    sample = pd.concat([sample, rich.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    bank_reserves.append(sample_std)

In [12]:
ecv_active = pd.read_csv("C:/Users/met48/Desktop/TS-Clustering/SimData/epsteinCV_outputs_active.csv", header=None)
ecv_jailed = pd.read_csv("C:/Users/met48/Desktop/TS-Clustering/SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_quiet = pd.read_csv("C:/Users/met48/Desktop/TS-Clustering/SimData/epsteinCV_outputs_quiet.csv", header=None)
sc = StandardScaler()
ecv = []
for i in np.arange(0, ecv_active.shape[0]):
    sample = pd.concat([ecv_active.iloc[i], ecv_jailed.iloc[i]], axis=0).T
    sample = pd.concat([sample, ecv_quiet.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ecv.append(sample_std)

In [6]:
def import_ff_data(filename):
    expected_columns=155
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split(',')
            if len(row) < expected_columns:
                row += [np.nan] * (expected_columns - len(row))
            data.append(row)
    df = pd.DataFrame(data)
    def fill_last_valid(row):
        for i in range(1, len(row)):
            if pd.isna(row[i]):
                row[i] = row[i - 1]  
        return row
    df_filled = df.apply(fill_last_valid, axis=1)
    return df_filled

In [7]:
ff_onfire = import_ff_data("C:/Users/met48/Desktop/TS-Clustering/SimData/forest_fire_outputs_onfire.csv")
print("check 1")
ff_fine = import_ff_data("C:/Users/met48/Desktop/TS-Clustering/SimData/forest_fire_outputs_fine.csv")
print("check 2")
ff_burned = import_ff_data("C:/Users/met48/Desktop/TS-Clustering/SimData/forest_fire_outputs_burned.csv")
sc = StandardScaler()
ff = []
for i in np.arange(0, ff_onfire.shape[0]):
    sample = pd.concat([ff_onfire.iloc[i], ff_fine.iloc[i]], axis=0).T
    sample = pd.concat([sample, ff_burned.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ff.append(sample_std)

check 1
check 2


In [8]:
# Silouhette score 

from sklearn.metrics import silhouette_score
def sil_score(X_embed, labels, s): 
    SILH_SAMPLE = s 
    RANDOM_STATE = 42

    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="euclidean")
    else:
        sil = silhouette_score(X_embed, labels, metric="euclidean")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    print(f"[Kmeans, DAE] Silhouette: {sil:.4f}")
#    print("Cluster sizes:", cluster_sizes)
    return sil

In [9]:
import tensorflow as tf

def DAE_reduction(df, bottleneck, ep, batch):
    df_flat = [sample.flatten() for sample in df]
    df_flat = np.asarray(df_flat)
    train, test = train_test_split(df_flat, test_size=0.20, random_state=42)
    input = Input(shape=(df_flat.shape[1],))

    encoded = Dense(30, activation='relu')(input)
    encoded = Dense(20, activation='relu')(encoded)
    encoded = Dense(10, activation='relu')(encoded)
    encoded = Dense(bottleneck, activation='linear', name='bottleneck')(encoded)

    decoded = Dense(bottleneck, activation='relu')(encoded)
    decoded = Dense(20, activation='relu')(decoded)
    decoded = Dense(30, activation='relu')(decoded)
    output = Dense(df_flat.shape[1], activation=None)(decoded)

    autoencoder = Model(input, output)
#   autoencoder.summary()
#    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    huber = tf.keras.losses.Huber()
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss=huber)
    autoencoder.fit(train, train,
     epochs=ep,
     batch_size=batch,
     shuffle=True,
     validation_data=(test, test), verbose = 0)
    encoder = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('bottleneck').output)
    encoded_ts = encoder.predict(df_flat)
    return encoded_ts, encoder 

In [21]:
dae_embeds, dae_model = DAE_reduction(bank_reserves, 3, 3, 2048)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 598us/step


In [13]:
#br_dae = DAE_reduction(bank_reserves)
raw_data = []
raw_data.append(bank_reserves)
raw_data.append(ecv)
raw_data.append(ff)

In [14]:
opt_clusters = []
opt_clusters.append(7)
opt_clusters.append(8)
opt_clusters.append(4)

In [15]:
# Hyperparameter tuning; 
# hardcoded for DAE with linear bottleneck layer, 2048 batch size and Huber loss; 
# may want to tune them as well

def hypertune(target, k): 
    
    tf.keras.backend.clear_session()
    
    BOTTLENECK_MIN = 3 
    EPOCH_MIN = 2
    
    bottleneck = 9     #  number of desired embeddings; tune in the range of [BOTTLENECK_MIN..bottleneck]
    epochs = 8         #  tune in range of [EPOCH_MIN..epochs] 
    batch = 2048       #  fixed from prior trials 
    evals = 5          #  number of repeated trainings to reduce noise in model 
  
    sil_sample = 20000 # silohouette sample 

    max_sil_score = 0         # max silohouette score 
    best_embed = []           # best emebeddings 
    best_model = []   # model that produced the best embeddings 
    
    for b in np.arange(BOTTLENECK_MIN, bottleneck + 1):   # bottleneck 
        for e in np.arange(EPOCH_MIN, epochs + 1):        # epoch
            for i in np.arange(1, evals + 1):             # evals 
                
                print(f"Evaluation: embeddings {b}, epoch {e}, eval # {i}:")
                # get embeddings 
                dae_embeds, dae_model = DAE_reduction(raw_data[target], b, e, batch)

                # clustering 
                dae_kmeans_labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit(dae_embeds).labels_

                # silhouette score 
                score = sil_score(dae_embeds, dae_kmeans_labels, sil_sample)

                # save the best embeddings and model   
                if (score > max_sil_score): 
                    max_sil_score = score
                    best_embed = dae_embeds
                    best_model.append(dae_model) # assuming we are hypertuning the three models in order  
                    best_params = str(b) + '_' + str(e) + '_' + str(batch) + '_linear'
 
    print(f"Silhouette: {max_sil_score:.4f}")
    print(f"Best parameters: {best_params}")
    filename = 'C:/Users/met48/Desktop/TS-Clustering/extracted_features/' + str(target) + '_DAE_' + best_params + '.csv'
    np.savetxt(filename, best_embed, delimiter=",")

In [25]:
hypertune(0,7)

Evaluation: embeddings 3, epoch 2, eval # 1:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 610us/step
[Kmeans, DAE] Silhouette: 0.5209
Evaluation: embeddings 3, epoch 2, eval # 2:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 593us/step
[Kmeans, DAE] Silhouette: 0.4113
Evaluation: embeddings 3, epoch 2, eval # 3:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 571us/step
[Kmeans, DAE] Silhouette: 0.5654
Evaluation: embeddings 3, epoch 2, eval # 4:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 562us/step
[Kmeans, DAE] Silhouette: 0.5831
Evaluation: embeddings 3, epoch 2, eval # 5:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 661us/step
[Kmeans, DAE] Silhouette: 0.4461
Evaluation: embeddings 3, epoch 3, eval # 1:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 590us/step
[Kmeans, DAE] Silhouette: 0.5696
Evaluation: embeddings 3, epoch 3, eval # 2:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 611us/step
[Kmeans, DAE] Silhouette: 0.4889
Evaluation: embeddings 3, epoch 3, eval # 3:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 587us/step
[Kmeans, DAE] Silhouette: 0.5148
Evaluation: embe

In [26]:
hypertune(1, 8)

Evaluation: embeddings 3, epoch 2, eval # 1:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 637us/step
[Kmeans, DAE] Silhouette: 0.7109
Evaluation: embeddings 3, epoch 2, eval # 2:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 686us/step
[Kmeans, DAE] Silhouette: 0.7276
Evaluation: embeddings 3, epoch 2, eval # 3:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 643us/step
[Kmeans, DAE] Silhouette: 0.6722
Evaluation: embeddings 3, epoch 2, eval # 4:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 659us/step
[Kmeans, DAE] Silhouette: 0.7475
Evaluation: embeddings 3, epoch 2, eval # 5:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 626us/step
[Kmeans, DAE] Silhouette: 0.6535
Evaluation: embeddings 3, epoch 3, eval # 1:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 649us/step
[Kmeans, DAE] Silhouette: 0.6453
Evaluation: embeddings 3, epoch 3, eval # 2:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 1s 684us/step
[Kmeans, DAE] Silhouette: 0.6304
Evaluation: embeddings 3, epoch 3, eval # 3:
1574/1574 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
[Kmeans, DAE] Silhouette: 0.6054
Evaluation: embedd

In [16]:
hypertune(2, 4)


Evaluation: embeddings 3, epoch 2, eval # 1:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 630us/step


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\met48\AppData\Local\anaconda3\Lib\subprocess.py",

[Kmeans, DAE] Silhouette: 0.7887
Evaluation: embeddings 3, epoch 2, eval # 2:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 661us/step
[Kmeans, DAE] Silhouette: 0.8098
Evaluation: embeddings 3, epoch 2, eval # 3:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 662us/step
[Kmeans, DAE] Silhouette: 0.7931
Evaluation: embeddings 3, epoch 2, eval # 4:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 656us/step
[Kmeans, DAE] Silhouette: 0.9601
Evaluation: embeddings 3, epoch 2, eval # 5:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 671us/step
[Kmeans, DAE] Silhouette: 0.8164
Evaluation: embeddings 3, epoch 3, eval # 1:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 675us/step
[Kmeans, DAE] Silhouette: 0.7567
Evaluation: embeddings 3, epoch 3, eval # 2:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 625us/step
[Kmeans, DAE] Silhouette: 0.9307
Evaluation: embeddings 3, epoch 3, eval # 3:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 643us/step
[Kmeans, DAE] Silhouette: 0.7880
Evaluation: embeddings 3, epoch 3, eval # 4:
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2s 652us/step
[Kmeans, DAE] Si

In [ ]:
dae_embeds, dae_model = DAE_reduction(bank_reserves, 5, 2, 2048)
dae_kmeans_labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit(dae_embeds).labels_
score = sil_score(dae_embeds, dae_kmeans_labels, sil_sample)